In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
ROOT = Path.cwd().parent   # nbconvert and Jupyter both start the kernel in notebooks/
assert (ROOT / "paths.py").is_file(), f"paths.py not found from {Path.cwd()}"
sys.path.insert(0, str(ROOT))

# Sensitivity run: `ti_scale` lower bound 0.5 (not the headline)

Requested by the user (`closed_ideas.md`, reopen condition for the `ti_scale` bound). The headline fit keeps the spec bounds [0.7, 1.3]; this run widens only the `ti_scale` lower bound to 0.5 and repeats:

1. the Step 4 leave-one-block-out fit of (g_diff, f_ir, ti_scale), and
2. the Step 5 residual model on top of the new fold baselines, with the same 4 configurations and learner settings. The configuration is **not** re-selected: `no_elev` stays the reported one, and the others are shown for reference.

Everything else (fitting rows, exclusions, benchmark, scoring) is identical to Steps 4 and 5. Results are compared against `outputs/model_scores/1_physics_baseline/` and `outputs/model_scores/2_ground_correction/`.

In [ ]:
# --- config ---
import time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import paths
import thermal1d as t
import residual as r

DRY_RUN = False                # True: fit stubbed to the clipped defaults, for a fast end-to-end check
SUSPECT_SOLS = (3763, 3778)
EXPECTED_TRAIN_BLOCKS = [0, 1, 2, 3, 5, 6, 7, 9, 10]
LS_BIN_DEG = 30
REPORTED_CONFIG = "no_elev"   # Step 5 choice, kept fixed
BOUNDS = t.PARAM_BOUNDS.copy()
BOUNDS[t.PARAM_NAMES.index("ti_scale"), 0] = t.TI_SCALE_LO_SENSITIVITY
HEADLINE_FITS = paths.OUT_DIR / "fit_params.csv"
HEADLINE_RESID_SUMMARY = paths.RESIDUAL_OUT_DIR / "residual_summary.csv"
HEADLINE_RESID_BLOCKS = paths.RESIDUAL_OUT_DIR / "residual_by_block.csv"
HEADLINE_RESID_HOURS = paths.RESIDUAL_OUT_DIR / "residual_by_ltst_hour.csv"
HEADLINE_ROWS = paths.RESIDUAL_OUT_DIR / "residual_rows.csv"
OUT = paths.TI_SENS_OUT_DIR / ("dry_run" if DRY_RUN else "")
OUT.mkdir(parents=True, exist_ok=True)
PROGRESS_LOG = OUT / "ti_sens_progress.log"
PROGRESS_LOG.write_text("")
CONFIGS = r.CONFIGS
print("bounds:", dict(zip(t.PARAM_NAMES, BOUNDS.tolist())), "| dry run:", DRY_RUN)

## Data and evaluator (as in Steps 4 and 5)

In [ ]:
tbl = pd.DataFrame(t.load_rems_table(paths.DATA_DIR / "gale_rems_training.csv"))
assert (tbl.split_role == "train").all()
blocks = sorted(tbl.block.unique())
assert blocks == EXPECTED_TRAIN_BLOCKS, f"training blocks {blocks}"

per_sol_cols = ["ls_deg", "tau", "tau_is_placeholder", "row", "slope_deg", "aspect_deg",
                "albedo_mean", "ti_themis_tiu", "block"]
assert (tbl.groupby("sol")[per_sol_cols].nunique() == 1).all().all(), "per-sol inputs vary within a sol"
sols = tbl.groupby("sol")[per_sol_cols].first().reset_index()
sol_inputs = {"lat_deg": t.lat_from_row(sols.row.to_numpy()), "ls_deg": sols.ls_deg.to_numpy(),
              "tau": sols.tau.to_numpy(), "slope_deg": sols.slope_deg.to_numpy(),
              "aspect_deg": sols.aspect_deg.to_numpy(), "albedo": sols.albedo_mean.to_numpy(),
              "ti": sols.ti_themis_tiu.to_numpy()}
row_sol = pd.Series(np.arange(len(sols)), index=sols.sol)[tbl.sol].to_numpy()
row_step = t.bin_centre_step(tbl.bin.to_numpy())
evaluate = t.make_evaluator(sol_inputs, row_sol, row_step)

obs = tbl.ground_temp_k.to_numpy()
block = tbl.block.to_numpy()
suspect = tbl.sol.between(*SUSPECT_SOLS).to_numpy()
placeholder = tbl.tau_is_placeholder.to_numpy()
fit_ok = ~suspect & ~placeholder
tbl["group"] = np.where(suspect, "suspect_terrain", np.where(placeholder, "placeholder_tau", "main"))
tbl["ltst_hour"] = (tbl.bin // 4).astype(int)
tbl["ltst_hr"] = (tbl.bin + 0.5) * t.BIN_HOURS
tbl["ls_bin"] = (tbl.ls_deg // LS_BIN_DEG * LS_BIN_DEG).astype(int)
print(len(sols), "sols,", len(tbl), "rows;", int(fit_ok.sum()), "rows usable for fitting")

headline_rows = pd.read_csv(HEADLINE_ROWS)
assert (headline_rows.sol.to_numpy() == tbl.sol.to_numpy()).all() and \
    (headline_rows.bin.to_numpy() == tbl.bin.to_numpy()).all(), "row order differs from Step 5"

## Step 4 repeated: fold fits with the wider bound

In [ ]:
def fit(use):
    if DRY_RUN:
        p = np.clip(np.array([t.G_DIFF, t.F_IR, t.TI_SCALE]), BOUNDS[:, 0], BOUNDS[:, 1])
        return p, float(((evaluate(p) - obs)[use] ** 2).sum()), np.zeros(3, bool)
    return t.fit_global_params(evaluate, obs, use, bounds=BOUNDS)


def fit_row(label, p, sse, at_bound, n_fit):
    return {"fold": label, **dict(zip(t.PARAM_NAMES, p)),
            **{f"{n}_at_bound": bool(a) for n, a in zip(t.PARAM_NAMES, at_bound)},
            "fit_rows": n_fit, "fit_rmse_k": np.sqrt(sse / n_fit)}


fits = []
t0 = time.time()
for b in blocks + ["all"]:
    use = fit_ok & (block != b) if b != "all" else fit_ok
    label = f"block {b}" if b != "all" else "all training blocks"
    p, sse, at_bound = fit(use)
    fits.append(fit_row(label, p, sse, at_bound, int(use.sum())))
    line = (f"{label}: " + ", ".join(f"{n}={v:.4f}{' (BOUND)' if a else ''}" for n, v, a in zip(t.PARAM_NAMES, p, at_bound))
            + f" | fit RMSE {np.sqrt(sse / use.sum()):.2f} K | {time.time() - t0:.0f} s")
    print(line)
    with open(PROGRESS_LOG, "a") as fh:   # nbconvert shows output only at the end
        fh.write(line + "\n")
fits = pd.DataFrame(fits).set_index("fold")
fits.to_csv(OUT / "fit_params.csv")
head = pd.read_csv(HEADLINE_FITS).set_index("fold")
cmp = pd.concat({"headline": head[list(t.PARAM_NAMES) + ["fit_rmse_k"]],
                 "ti_lo_0.5": fits[list(t.PARAM_NAMES) + ["fit_rmse_k"]]}, axis=1)
print()
print(cmp.round(4).to_string())

## Step 5 repeated: residual model on the new fold baselines

In [ ]:
fold_base, fold_q = {}, {}
for label in fits.index:
    p = tuple(fits.loc[label, list(t.PARAM_NAMES)].to_numpy(dtype=float))
    g_diff, f_ir, _ = p
    q = t.absorbed_flux(sol_inputs["lat_deg"], sol_inputs["ls_deg"], sol_inputs["tau"],
                        sol_inputs["slope_deg"], sol_inputs["aspect_deg"], sol_inputs["albedo"],
                        t.step_ltst(), g_diff=g_diff, f_ir=f_ir)
    fold_base[label], fold_q[label] = evaluate(p).copy(), q[row_sol, row_step].copy()
    assert np.all(np.isfinite(fold_base[label])), f"non-finite baseline in {label}"

oof = np.full(len(tbl), np.nan)
tbl["bench_k"] = np.nan
for c in CONFIGS:
    tbl[f"pred_{c}"] = np.nan
for b in blocks:
    label = f"block {b}"
    held = block == b
    train = fit_ok & ~held
    oof[held] = fold_base[label][held]
    frame = tbl.assign(baseline_k=fold_base[label], q_abs_wm2=fold_q[label])
    X = r.make_features(frame)
    target = obs - fold_base[label]
    for c, cols in CONFIGS.items():
        model = r.make_learner().fit(X.loc[train, cols], target[train])
        tbl.loc[held, f"pred_{c}"] = fold_base[label][held] + model.predict(X.loc[held, cols])
    tbl.loc[held, "bench_k"] = r.benchmark(tbl[~held], tbl[held])[0]
tbl["baseline_k"] = oof
assert tbl.filter(like="pred_").notna().all().all() and tbl.bench_k.notna().all()
assert np.abs(tbl.bench_k.to_numpy() - headline_rows.bench_k.to_numpy()).max() < 1e-9, "benchmark differs from Step 5"

## Scores, side by side with the headline (spec bounds)

In [ ]:
MODELS = ["baseline", "bench"] + list(CONFIGS)
COLS = {"baseline": "baseline_k", "bench": "bench_k", **{c: f"pred_{c}" for c in CONFIGS}}
for m in MODELS:
    tbl[f"resid_{m}"] = tbl[COLS[m]] - tbl.ground_temp_k


def score(g, models=MODELS):
    out = {"n": len(g)}
    for m in models:
        out[f"{m}_rmse_k"] = np.sqrt((g[f"resid_{m}"] ** 2).mean())
        out[f"{m}_bias_k"] = g[f"resid_{m}"].mean()
    return pd.Series(out)


scored = tbl[tbl.group != "placeholder_tau"]
main = tbl[tbl.group == "main"]
by_block = scored.groupby("block").apply(score, include_groups=False)
by_group = tbl.groupby("group").apply(score, include_groups=False)
summary = pd.DataFrame({
    "macro_rmse_k": {m: by_block[f"{m}_rmse_k"].mean() for m in MODELS},
    "macro_rmse_no_3763_k": {m: by_block.drop(index=10)[f"{m}_rmse_k"].mean() for m in MODELS},
    "pooled_rmse_main_k": {m: np.sqrt((main[f"resid_{m}"] ** 2).mean()) for m in MODELS},
    "rmse_placeholder_tau_k": {m: by_group.loc["placeholder_tau", f"{m}_rmse_k"] for m in MODELS},
    "rmse_suspect_terrain_k": {m: by_group.loc["suspect_terrain", f"{m}_rmse_k"] for m in MODELS},
    "blocks_beating_bench": {m: int((by_block[f"{m}_rmse_k"] < by_block.bench_rmse_k).sum()) for m in MODELS},
})
head_summary = pd.read_csv(HEADLINE_RESID_SUMMARY, index_col=0)
head_blocks = pd.read_csv(HEADLINE_RESID_BLOCKS, index_col=0)
pd.options.display.float_format = "{:.2f}".format
print("Summary, headline (spec bounds) vs sensitivity (ti_scale >= 0.5):")
print(pd.concat({"headline": head_summary, "ti_lo_0.5": summary}, axis=0).to_string())
print()
keep = ["baseline", "bench", REPORTED_CONFIG]
blk = pd.concat({"headline": head_blocks[[f"{m}_rmse_k" for m in keep] + [f"{m}_bias_k" for m in keep]],
                 "ti_lo_0.5": by_block[[f"{m}_rmse_k" for m in keep] + [f"{m}_bias_k" for m in keep]]}, axis=1)
print("Per block:")
print(blk.to_string())

In [ ]:
head_rows = headline_rows.assign(resid_baseline=headline_rows.baseline_k - headline_rows.ground_temp_k,
                                 resid_no_elev=headline_rows.pred_no_elev - headline_rows.ground_temp_k)
hours = {}
for name, d in (("headline", head_rows), ("ti_lo_0.5", tbl)):
    for subset, mask in (("main, blocks other than 0", (d.group == "main") & (d.block != 0)),
                         ("block 0", d.block == 0)):
        g = d[mask].groupby("ltst_hour")
        hours[(name, subset, "baseline_bias_k")] = g.resid_baseline.mean()
        hours[(name, subset, "no_elev_bias_k")] = g.resid_no_elev.mean()
        hours[(name, subset, "no_elev_rmse_k")] = np.sqrt(g.resid_no_elev.apply(lambda x: (x ** 2).mean()))
hours = pd.DataFrame(hours)
night = hours.index.isin([0, 1, 2, 3, 4, 5, 19, 20, 21, 22, 23])
print("Night (19-06 h) mean bias and RMSE (K):")
print(hours[night].mean().unstack(-1).to_string())
print()
print("By LTST hour:")
print(hours.round(2).to_string())

tbl.to_csv(OUT / "ti_sens_rows.csv", index=False)
summary.to_csv(OUT / "ti_sens_summary.csv")
by_block.to_csv(OUT / "ti_sens_by_block.csv")
hours.to_csv(OUT / "ti_sens_by_ltst_hour.csv")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, subset in zip(axes[:2], ["main, blocks other than 0", "block 0"]):
    for name, st in (("headline", "--"), ("ti_lo_0.5", "-")):
        ax.plot(hours.index, hours[(name, subset, "baseline_bias_k")], "r" + st, label=f"baseline, {name}")
        ax.plot(hours.index, hours[(name, subset, "no_elev_bias_k")], "g" + st, label=f"+ residual, {name}")
    ax.axhline(0, color="grey", lw=0.8)
    ax.set(xlabel="LTST hour", ylabel="bias (K)", title=f"Bias by LTST hour: {subset}")
    ax.legend(fontsize=7)
ax = axes[2]
x = np.arange(len(by_block))
ax.bar(x - 0.3, head_blocks[f"{REPORTED_CONFIG}_rmse_k"], 0.3, label=f"{REPORTED_CONFIG}, headline")
ax.bar(x, by_block[f"{REPORTED_CONFIG}_rmse_k"], 0.3, label=f"{REPORTED_CONFIG}, ti_lo_0.5")
ax.bar(x + 0.3, by_block.bench_rmse_k, 0.3, label="benchmark")
ax.set_xticks(x, [str(b) for b in by_block.index])
ax.set(xlabel="held-out block", ylabel="RMSE (K)", title="LOBO RMSE")
ax.legend(fontsize=7)
fig.tight_layout()
fig.savefig(OUT / "ti_sens_scores.png", dpi=120)
print("saved to", OUT)